# 04 — Compound Hazard Detection
**Project:** GB Compound Hazards Risk Prioritization

**Course:** Artificial Intelligence in Human Water Systems

---
## Objectives
- Identify spatiotemporal overlaps between precipitation and wind clusters
- Define compound hazard events following Tilloy et al. (2022)
- Assign compound events to NUTS1 regions of Great Britain
- Save compound hazard database for risk categorization

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load Cluster Databases

In [ ]:
rain_db = pd.read_parquet('../data/processed/rain_clusters_db.parquet')
wind_db = pd.read_parquet('../data/processed/wind_clusters_db.parquet')

print(f'Precipitation clusters: {len(rain_db):,}')
print(f'Wind clusters:          {len(wind_db):,}')
print('\nColumns:', list(rain_db.columns))

## 3. Detect Compound Hazard Overlaps

A compound hazard event is identified when:
1. **Temporal overlap:** the time windows of a rain cluster and wind cluster overlap
2. **Spatial overlap:** the bounding boxes of both clusters intersect

Following Tilloy et al. (2022):
- Compound duration = union of both durations (OR)
- Compound footprint = intersection of both footprints (AND)

In [ ]:
def check_spatial_overlap(r_lat_min, r_lat_max, r_lon_min, r_lon_max,
                           w_lat_min, w_lat_max, w_lon_min, w_lon_max):
    lat_overlap = r_lat_min <= w_lat_max and w_lat_min <= r_lat_max
    lon_overlap = r_lon_min <= w_lon_max and w_lon_min <= r_lon_max
    return lat_overlap and lon_overlap

compound_events = []
compound_id = 0

print(f'Checking {len(rain_db):,} precipitation x {len(wind_db):,} wind clusters...')

for _, rain_row in tqdm(rain_db.iterrows(), total=len(rain_db), desc='Processing rain clusters'):
    temporal_candidates = wind_db[
        (wind_db['start_time'] <= rain_row['end_time']) &
        (wind_db['end_time'] >= rain_row['start_time'])
    ]

    for _, wind_row in temporal_candidates.iterrows():
        if not check_spatial_overlap(
            rain_row['lat_min'], rain_row['lat_max'],
            rain_row['lon_min'], rain_row['lon_max'],
            wind_row['lat_min'], wind_row['lat_max'],
            wind_row['lon_min'], wind_row['lon_max']
        ):
            continue

        compound_start = min(rain_row['start_time'], wind_row['start_time'])
        compound_end   = max(rain_row['end_time'], wind_row['end_time'])
        compound_duration_h = (compound_end - compound_start).total_seconds() / 3600

        inter_lat_min = max(rain_row['lat_min'], wind_row['lat_min'])
        inter_lat_max = min(rain_row['lat_max'], wind_row['lat_max'])
        inter_lon_min = max(rain_row['lon_min'], wind_row['lon_min'])
        inter_lon_max = min(rain_row['lon_max'], wind_row['lon_max'])

        lat_span = inter_lat_max - inter_lat_min
        lon_span = inter_lon_max - inter_lon_min
        n_cells_approx = max(1, int((lat_span / 0.25) * (lon_span / 0.25)))
        cell_area_km2 = (0.25 * 111) * (0.25 * 65)
        compound_extent_km2 = n_cells_approx * cell_area_km2

        month = compound_start.month
        if month in [12, 1, 2]:   season = 'DJF'
        elif month in [3, 4, 5]:  season = 'MAM'
        elif month in [6, 7, 8]:  season = 'JJA'
        else:                      season = 'SON'

        compound_events.append({
            'compound_id': compound_id,
            'rain_cluster_id': rain_row['cluster_id'],
            'wind_cluster_id': wind_row['cluster_id'],
            'compound_start': compound_start,
            'compound_end': compound_end,
            'compound_duration_h': compound_duration_h,
            'compound_extent_km2': compound_extent_km2,
            'peak_precipitation_mm': rain_row['peak_intensity'],
            'peak_wind_ms': wind_row['peak_intensity'],
            'mean_precipitation_mm': rain_row['mean_intensity'],
            'mean_wind_ms': wind_row['mean_intensity'],
            'rain_duration_h': rain_row['duration_h'],
            'wind_duration_h': wind_row['duration_h'],
            'lat_min': inter_lat_min,
            'lat_max': inter_lat_max,
            'lon_min': inter_lon_min,
            'lon_max': inter_lon_max,
            'lat_centroid': (inter_lat_min + inter_lat_max) / 2,
            'lon_centroid': (inter_lon_min + inter_lon_max) / 2,
            'year': compound_start.year,
            'season': season
        })
        compound_id += 1

compound_db = pd.DataFrame(compound_events)
print(f'\nCompound hazard events detected: {len(compound_db):,}')

## 4. Assign NUTS1 Regions

In [ ]:
NUTS1_REGIONS = {
    'Scotland':                  (54.6, 60.0, -7.0, -0.5),
    'Wales':                     (51.3, 53.4, -5.3, -2.6),
    'North East England':        (54.5, 55.8, -2.4,  0.0),
    'North West England':        (53.3, 55.0, -3.4, -1.7),
    'Yorkshire and The Humber':  (53.3, 54.6, -2.6,  0.2),
    'East Midlands':             (52.2, 53.5, -1.8,  0.9),
    'West Midlands':             (51.9, 53.2, -2.7, -1.1),
    'East of England':           (51.5, 53.2,  0.0,  1.8),
    'London':                    (51.3, 51.7, -0.5,  0.3),
    'South East England':        (50.7, 51.9, -1.8,  1.8),
    'South West England':        (49.9, 51.8, -5.7, -1.8)
}

def assign_nuts1_region(lat_centroid, lon_centroid):
    for region, (lat_min, lat_max, lon_min, lon_max) in NUTS1_REGIONS.items():
        if lat_min <= lat_centroid <= lat_max and lon_min <= lon_centroid <= lon_max:
            return region
    return 'Outside GB'

compound_db['nuts1_region'] = compound_db.apply(
    lambda row: assign_nuts1_region(row['lat_centroid'], row['lon_centroid']),
    axis=1
)

print('Compound events per NUTS1 region:')
print(compound_db['nuts1_region'].value_counts().to_string())

## 5. Compound Events Summary

In [ ]:
print('=== Compound Hazard Events Summary ===')
print(compound_db[[
    'compound_duration_h', 'compound_extent_km2',
    'peak_precipitation_mm', 'peak_wind_ms'
]].describe().round(3))

print('\nCompound events per season:')
print(compound_db.groupby('season').size().to_string())

## 6. Visualize Compound Events per Region

In [ ]:
region_counts = compound_db[compound_db['nuts1_region'] != 'Outside GB']['nuts1_region'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(region_counts.index, region_counts.values, color='mediumpurple', edgecolor='white')
ax.set_title('Compound Hazard Events per NUTS1 Region\n(Great Britain, 2009-2019)', fontsize=13)
ax.set_xlabel('Number of compound events')
ax.bar_label(bars, padding=3)
plt.tight_layout()
plt.savefig('../outputs/figures/04_compound_events_per_region.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved!')

## 7. Save Compound Database

In [ ]:
compound_db.to_parquet('../data/processed/compound_hazards_db.parquet', index=False)
compound_db.to_csv('../data/processed/compound_hazards_db.csv', index=False)

print(f'Saved compound_hazards_db with {len(compound_db):,} events')
print('  data/processed/compound_hazards_db.parquet')
print('  data/processed/compound_hazards_db.csv')